# SatQuery AI — Division 2: Single-Image Remote-Sensing Intelligence
## Google Colab GPU Compute Pipeline & Scientific Training Runner

- **Division**: Division 2 (Single-Image Remote-Sensing Intelligence: VQA + Visual Grounding)
- **Lead Owner**: Sruthi (`sruthi-270` / `rajamanurisruthi@gmail.com`)
- **Branch**: `feature/sruthi-single-image`
- **Target Base Model**: `google/paligemma-3b-pt-224`
- **Status**: `[SCIENTIFIC EVALUATION NOT VERIFIED — AWAITING REAL GPU SMOKE TEST]`

> **Notice**: This notebook runs exclusively as an external GPU compute worker. The final SatQuery application runtime does not depend on Colab.

### Step 1: GPU Compute Environment & Hardware Diagnostics

In [26]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model:       {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total:      {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"CUDA Version:    {torch.version.cuda}")

Tue Aug 25 13:18:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P0             29W /   70W |    5721MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Step 2: Secure Hugging Face Authentication

In [27]:
import os
import huggingface_hub

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    import getpass
    hf_token = getpass.getpass('Enter Hugging Face Access Token (Read role): ')

huggingface_hub.login(token=hf_token)
print("Hugging Face authentication completed securely.")

Hugging Face authentication completed securely.


### Step 3: Fast Git Repository Checkout & Dependency Setup

In [28]:
import os, sys
if not os.path.exists('/content/SatQuery'):
    !git clone https://github.com/Lalith2007/SatQuery.git /content/SatQuery
%cd /content/SatQuery
!git fetch origin
!git checkout feature/sruthi-single-image
!git reset --hard origin/feature/sruthi-single-image

# Fix Colab torchao conflict and install dependencies
!pip uninstall -y torchao
!pip install fastapi pydantic-settings tifffile pytest-asyncio peft
!pip install -e . --no-deps

if '/content/SatQuery' not in sys.path:
    sys.path.insert(0, '/content/SatQuery')

print("✓ Environment and repository ready!")

/content/SatQuery
Already on 'feature/sruthi-single-image'
Your branch is up to date with 'origin/feature/sruthi-single-image'.
HEAD is now at 887501c fix(division-2): ensure robust processor tokenization, labels, and memory management
Obtaining file:///content/SatQuery
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for satquery (pyproject.toml) ... done
  Created wheel for satquery: filename=satquery-0.1.0-0.editable-py3-none-any.whl size=7617 sha256=454deec926b524adf3643426757cb5d1d442ee409adef4ab5f8eb1c50943187a
  Stored in directory: /tmp/pip-ephem-wheel-cache-zdpmz24r/wheels/cc/20/0d/d7f49facc33d95943a6069e43a7a4874a0b0eb4633eefa9848
Successfully built satquery
  Attempting uninstall: satquery
    Found existing installation: satquery 0.1.0
    Uninstalling satquery-0.1.0:
      Successfully uninstalled

### Step 4: Phase 1 — Real Model Load & Generation Verification

In [29]:
import torch, gc
from PIL import Image
from transformers import PaliGemmaForConditionalGeneration

model_id = "google/paligemma-3b-pt-224"

print(f"Loading {model_id} on GPU...")
try:
    from transformers import PaliGemmaProcessor
    processor = PaliGemmaProcessor.from_pretrained(model_id)
except Exception:
    from transformers import AutoProcessor
    processor = AutoProcessor.from_pretrained(model_id)

dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = PaliGemmaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=dtype,
    low_cpu_mem_usage=True,
    device_map="cuda:0" if torch.cuda.is_available() else None,
)

total_params = sum(p.numel() for p in model.parameters())
print(f"SUCCESS: Loaded {model_id} on {model.device}!")
print(f"Total Model Parameters: {total_params:,}")

# Test genuine model.generate() output
test_img = Image.new("RGB", (224, 224), color=(34, 139, 34))
inputs = processor(text="answer en What is the dominant land cover?", images=test_img, return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_new_tokens=32)
ans = processor.decode(output[0], skip_special_tokens=True)
print(f"Genuine Model Generation Output: '{ans}'")
print("PHASE 1 PASSED: REAL_MODEL_LOADED = True")

# Free Step 4 model memory so Step 5 has full GPU memory available
del model, processor
gc.collect()
torch.cuda.empty_cache()
print("GPU memory freed for Phase 2 training.")

Loading google/paligemma-3b-pt-224 on GPU...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/603 [00:00<?, ?it/s]

[transformers] You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.


SUCCESS: Loaded google/paligemma-3b-pt-224 on cuda:0!
Total Model Parameters: 2,923,466,480
Genuine Model Generation Output: 'answer en What is the dominant land cover?
grass'
PHASE 1 PASSED: REAL_MODEL_LOADED = True
GPU memory freed for Phase 2 training.


### Step 5: Phase 2 — Real LoRA Gradient & Backprop Smoke Test (Mandatory Proof)

In [30]:
# Runs genuine forward pass, loss.backward(), non-zero gradient check, and optimizer.step() parameter delta
!python3 specialists/single_image/adaptation/train_lora.py --smoke-test --device cuda

2026-08-25 13:19:58 | INFO     | [satquery.train_lora] [train_lora.py:88] | ================================================================================
2026-08-25 13:19:58 | INFO     | [satquery.train_lora] [train_lora.py:89] | PHASE 2: REAL PEFT / LORA GRADIENT & BACKPROPAGATION SMOKE TEST ON [CUDA]
2026-08-25 13:20:03 | INFO     | [satquery.train_lora] [train_lora.py:105] | Loading base PaliGemma: google/paligemma-3b-pt-224 (revision: main)...
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files: 100% 3/3 [00:00<00:00, 1053.58it/s]
Download complete: :           |  0.00B            
Download complete: :           |  0.00B            0B                         
Reconstruction complete: |          |  0.00B /  0.00B            
Loading weights: 100% 603/603 [00:46<00:00, 12.88it/s]
2026-08-25 13:20:55 | INFO     | [satquery.train_lora] [train_lora.py:139] | Total Parameters:     2,934,765,296
2026-08-25 13:20:55 | INFO     | [satquery.tra

### Step 6: Phase 3 — Real LoRA Domain Adaptation Training (5 Epochs on GPU)

In [ ]:
# Execute real gradient-based LoRA training across 900 training samples
!python3 specialists/single_image/adaptation/train_lora.py --epochs 5 --device cuda

2026-08-25 13:21:02 | INFO     | [satquery.train_lora] [train_lora.py:287] | ================================================================================
2026-08-25 13:21:02 | INFO     | [satquery.train_lora] [train_lora.py:288] | STARTING REAL LORA DOMAIN ADAPTATION TRAINING ON [CUDA]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files: 100% 3/3 [00:00<00:00, 1146.51it/s]
Download complete: :           |  0.00B            
Download complete: :           |  0.00B            0B                         
Reconstruction complete: |          |  0.00B /  0.00B            
Loading weights:   0% 0/603 [00:00<?, ?it/s]

### Step 7: Phase 4 & 5 — Real Model Evaluation & Synchronized CUDA Latency Benchmark

In [ ]:
# Strict scientific evaluation on N=150 held-out samples (Zero fallback permitted)
!python3 specialists/single_image/evaluation/reproducibility.py

2026-08-25 13:16:06 | INFO     | [satquery.reproducibility] [reproducibility.py:352] | Executing Division 2 Real Adaptation Scientific Verification Audit...
2026-08-25 13:16:06 | INFO     | [satquery.single_image_model] [model.py:98] | Loading PaliGemma RS engine on device: 'cuda' (Base: google/paligemma-3b-pt-224)...
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files: 100% 3/3 [00:00<00:00, 1082.12it/s]
Download complete: :           |  0.00B            
Download complete: :           |  0.00B            0B                         
Reconstruction complete: |          |  0.00B /  0.00B            
Loading weights: 100% 603/603 [00:45<00:00, 13.13it/s]
2026-08-25 13:17:02 | INFO     | [satquery.single_image_model] [model.py:130] | Successfully loaded REAL PaliGemma model with weights into memory.
[transformers] You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many to

### Step 8: Phase 6 — Export Reproducibility Manifest & Artifact Archive

In [ ]:
!python3 specialists/single_image/colab/reproducibility_manifest.py
!tar -czvf satquery_division2_adapter_package.tar.gz specialists/single_image/weights/ specialists/single_image/evaluation/ specialists/single_image/colab/
print("Artifact bundle generated: satquery_division2_adapter_package.tar.gz")